# Fair-UK evaluation before and after unlearning

Evaluate six base models and their five unlearning variants with Fair-UK. Collect baseline measurements, train with the method notebooks, then collect post-training measurements and compare them. Each baseline is shared across the five methods.

See the [evaluation setup](EVALUATION.md). Enable the execution switches for the stages you want to run.

In [ ]:
import csv
import hashlib
import html
import json
import math
import os
import subprocess
from pathlib import Path

from IPython.display import HTML, display

repo = next(
    p for p in (Path.cwd(), *Path.cwd().parents) if (p / "configs/unlearning/models.json").is_file()
)
settings_path = repo / "configs/unlearning/local.json"
settings = json.loads(settings_path.read_text()) if settings_path.exists() else {}
MODELS = json.loads((repo / "configs/unlearning/models.json").read_text())["models"]
METHODS = ("rmu", "task_vector", "wga_kl", "npo_kl", "biasunlearn")
MODELS_ROOT = (
    Path(os.environ.get("FAIRFORGET_MODELS_ROOT", settings.get("models_root", repo / "models")))
    .expanduser()
    .resolve()
)
TRAINING_ROOT = (
    Path(settings.get("output_root", repo / "outputs/unlearning")).expanduser().resolve()
)
TRAINING_ROOTS = {method: TRAINING_ROOT / method for method in METHODS}

## Evaluation setup

Set the Fair-UK checkout, evaluation Python executable, evaluation manifest and development manifest. `TASKS` lists the native Ukrainian benchmarks. To evaluate additional native views, add their task IDs from the evaluation manifest.

Edit `TRAINING_ROOTS` if the method outputs are stored in different locations. RMU exports are under `checkpoint`; the other methods use `merged_checkpoint`.

In [ ]:
FAIR_UK_ROOT = Path(os.environ.get("FAIR_UK_ROOT", repo.parent / "fair-uk")).expanduser().resolve()
FAIR_UK_PYTHON = (
    Path(os.environ.get("FAIR_UK_PYTHON", repo / ".venv-evaluation/bin/python"))
    .expanduser()
    .absolute()
)
EVALUATION_BUNDLE = (
    Path(os.environ.get("FAIR_UK_EVALUATION_BUNDLE", repo / "data/evaluation/manifest.json"))
    .expanduser()
    .resolve()
)
QUALIFICATION_BUNDLE = (
    Path(os.environ.get("FAIR_UK_QUALIFICATION_BUNDLE", repo / "data/development/manifest.json"))
    .expanduser()
    .resolve()
)
QUALIFICATION_TASK = "warbias_triplets_uk"
EVALUATION_ROOT = (
    Path(os.environ.get("FAIR_UK_OUTPUT", repo / "outputs/evaluation")).expanduser().resolve()
)
TASKS = (
    "stereoset_uk",
    "bbq_uk",
    "winobias_uk_natural",
    "winobias_uk_controlled",
    "winopron_uk",
    "warbias_triplets_uk",
)
RUN_BEFORE = False
RUN_AFTER = False
RUN_COMPARISON = False

# Shared settings: only the checkpoint path changes between before and after.
SEED = 42
BOOTSTRAP = 1000
ABSOLUTE_TOLERANCE = 1e-6
MODEL_SETTINGS = "dtype=bfloat16,tensor_parallel_size=1,max_model_len=4096,seed=42,enable_thinking=False,enforce_eager=True"
CLI_SETTINGS = [
    "--batch-size",
    "8",
    "--chunk-size",
    "128",
    "--max-output-tokens",
    "512",
    "--generation-prompt-format",
    "native-chat",
    "--answer-policy",
    "abc_option_text_v2",
    "--bootstrap",
    str(BOOTSTRAP),
    "--seed",
    str(SEED),
]
BASE_CHECKPOINTS = {
    model["id"]: MODELS_ROOT / model["directory"] / model["revision"] for model in MODELS
}
AFTER_CHECKPOINTS = {
    (method, model["id"]): TRAINING_ROOTS[method]
    / model["id"]
    / ("checkpoint" if method == "rmu" else "merged_checkpoint")
    for method in METHODS
    for model in MODELS
}
for model in MODELS:
    print(model["id"], "| base:", BASE_CHECKPOINTS[model["id"]])
print(
    "Models:",
    len(MODELS),
    "| methods:",
    len(METHODS),
    "| post checkpoints:",
    len(AFTER_CHECKPOINTS),
)

## Input validation

Input hashes and checkpoint exports are checked against the manifests and training receipts. Each collection writes to a new output directory. Keep the same evaluation root to reuse baseline runs in the after and comparison stages.

In [ ]:
def read_json(path):
    return json.loads(Path(path).read_text())


def sha256(path):
    value = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            value.update(block)
    return value.hexdigest()


def save_json(path, value):
    Path(path).write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False) + "\n")


def bundle_rows(bundle, tasks, *, development=False):
    manifest = read_json(bundle)
    if manifest.get("schema_version") != 1:
        raise ValueError("Expected a Fair-UK manifest with schema_version=1.")
    selected = {}
    for task in tasks:
        entry = manifest["datasets"][task]
        if entry.get("measurement_eligibility") != "active":
            raise ValueError(f"{task}: the source is not active for measurement.")
        if (
            development
            and entry.get("split") != "development"
            and entry.get("source_role") != "development"
        ):
            raise ValueError("The repeatability check requires development data.")
        source = (bundle.parent / entry["filename"]).resolve()
        if not source.is_relative_to(bundle.parent) or sha256(source) != entry["sha256"]:
            raise ValueError(f"{task}: source path or SHA256 differs.")
        rows = [json.loads(line) for line in source.read_text().splitlines() if line.strip()]
        ids = [row["id"] for row in rows]
        if not rows or len(rows) != entry["rows"] or len(set(ids)) != len(ids):
            raise ValueError(f"{task}: incomplete or duplicate source IDs.")
        selected[task] = (entry, rows)
    return selected


def require_checkpoint(path):
    if not (path / "config.json").is_file() or not (path / "tokenizer_config.json").is_file():
        raise ValueError(f"Checkpoint or tokenizer is missing: {path}")
    if (path / "adapter_config.json").exists() or not any(path.glob("*.safetensors")):
        raise ValueError(f"Use a full self-contained checkpoint, not an adapter: {path}")
    if any(char in str(path) for char in (",", "\n", "\r")):
        raise ValueError("Checkpoint paths cannot contain commas or newlines.")


def require_training(method, model):
    directory = TRAINING_ROOTS[method] / model["id"]
    receipt_path = directory / ("run.json" if method == "rmu" else "intervention.json")
    receipt = read_json(receipt_path)
    expected = "trained_not_evaluated" if method == "rmu" else "trained_merged"
    identity = receipt["config"] if method == "rmu" else receipt
    if (
        receipt.get("method") != method
        or receipt.get("status") != expected
        or receipt.get("steps", 0) < 1
    ):
        raise ValueError(f"{method}/{model['id']}: training is incomplete.")
    if (
        identity.get("model_id") != model["repository"]
        or identity.get("model_revision") != model["revision"]
    ):
        raise ValueError("The training receipt names a different base model.")
    checkpoint = AFTER_CHECKPOINTS[method, model["id"]]
    declared = receipt["checkpoint"] if method == "rmu" else receipt["merged_checkpoint"]
    if Path(declared).resolve() != checkpoint.resolve():
        raise ValueError("The checkpoint path differs from its training receipt.")
    require_checkpoint(checkpoint)
    inventory = (
        receipt["checkpoint_identity"] if method == "rmu" else receipt["merged_checkpoint_files"]
    )
    if not isinstance(inventory, list) or not inventory:
        raise ValueError("The trainer did not record its exported checkpoint files.")
    for item in inventory:
        path = (checkpoint / item["path"]).resolve()
        if (
            not path.is_relative_to(checkpoint.resolve())
            or not path.is_file()
            or path.stat().st_size != item["bytes"]
            or sha256(path) != item["sha256"]
        ):
            raise ValueError(f"Trained checkpoint bytes differ from the receipt: {path}")
    return {
        "receipt": str(receipt_path),
        "sha256": sha256(receipt_path),
        "checkpoint": str(checkpoint),
    }


def preflight():
    if len(MODELS) != 6 or len(AFTER_CHECKPOINTS) != 30:
        raise ValueError("Expected six models and five method configurations.")
    if not FAIR_UK_PYTHON.is_file() or not (FAIR_UK_ROOT / "lm_eval/fair_uk/__main__.py").is_file():
        raise ValueError("Configure the installed Fair-UK checkout and evaluation Python.")
    full = bundle_rows(EVALUATION_BUNDLE, TASKS)
    dev = bundle_rows(QUALIFICATION_BUNDLE, [QUALIFICATION_TASK], development=True)
    if QUALIFICATION_TASK in full:
        dev_ids = {row["id"] for row in dev[QUALIFICATION_TASK][1]}
        if dev_ids & {row["id"] for row in full[QUALIFICATION_TASK][1]}:
            raise ValueError("Development and evaluation item IDs overlap.")
    protected = [
        FAIR_UK_ROOT,
        EVALUATION_BUNDLE.parent,
        QUALIFICATION_BUNDLE.parent,
        *BASE_CHECKPOINTS.values(),
        *TRAINING_ROOTS.values(),
    ]
    if any(
        EVALUATION_ROOT == path.resolve() or EVALUATION_ROOT.is_relative_to(path.resolve())
        for path in protected
    ):
        raise ValueError("Choose an evaluation output outside all source and training inputs.")
    return full, dev

## Fair-UK commands

Collection uses `python -m lm_eval.fair_uk run`. Comparison uses `paper compare-native`, which checks paired source items, tokenizer inputs and inference settings. Each checkpoint runs in a separate process.

In [ ]:
def invoke(arguments, destination, *, gpu=True):
    if destination.exists():
        raise FileExistsError(f"Output directory already exists: {destination}")
    destination.parent.mkdir(parents=True, exist_ok=True)
    log_path = destination.parent / (destination.name + ".log")
    invocation_path = destination.parent / (destination.name + ".command.json")
    if log_path.exists() or invocation_path.exists():
        raise FileExistsError(f"Command log already exists: {log_path}")
    command = [str(FAIR_UK_PYTHON), "-m", "lm_eval.fair_uk", *arguments]
    environment = dict(os.environ)
    environment.update(
        PYTHONPATH=str(FAIR_UK_ROOT),
        HF_HUB_OFFLINE="1",
        TRANSFORMERS_OFFLINE="1",
        HF_DATASETS_OFFLINE="1",
        HF_HUB_DISABLE_IMPLICIT_TOKEN="1",
        VLLM_NO_USAGE_STATS="1",
    )
    if not gpu:
        environment["CUDA_VISIBLE_DEVICES"] = ""
    save_json(invocation_path, {"command": command, "fair_uk_root": str(FAIR_UK_ROOT)})
    with log_path.open("x") as log:
        completed = subprocess.run(
            command,
            cwd=FAIR_UK_ROOT,
            env=environment,
            stdout=log,
            stderr=subprocess.STDOUT,
            check=False,
        )
    if completed.returncode:
        raise RuntimeError(f"Fair-UK failed; see {log_path}")


def collect(checkpoint, task, bundle, destination):
    invoke(
        [
            "run",
            "--task",
            task,
            "--dataset-bundle",
            str(bundle),
            "--model",
            "vllm",
            "--model-args",
            f"pretrained={checkpoint},{MODEL_SETTINGS}",
            *CLI_SETTINGS,
            "--output",
            str(destination),
        ],
        destination,
    )


def require_native(directory, task, checkpoint, bundle):
    entry = read_json(bundle)["datasets"][task]
    record = read_json(directory / "run.json")
    read_json(directory / "report.json")
    predictions = [
        json.loads(line)
        for line in (directory / "predictions.jsonl").read_text().splitlines()
        if line.strip()
    ]
    ids = [row["id"] for row in predictions]
    checkpoint_path = record.get("model_identity", {}).get("parameters", {}).get("pretrained", "")
    if (
        record.get("task") != task
        or record.get("selected_rows") != entry["rows"]
        or record.get("limit_clusters_per_stratum") is not None
        or record.get("dataset", {}).get("sha256") != entry["sha256"]
        or len(ids) != entry["rows"]
        or len(set(ids)) != len(ids)
        or hashlib.sha256("\n".join(ids).encode()).hexdigest() != record.get("selected_ids_sha256")
        or Path(checkpoint_path).resolve() != checkpoint.resolve()
    ):
        raise ValueError(f"Incomplete or mismatched Fair-UK collection: {directory}")


def qualify(checkpoint, label, development):
    # Repeat the same checkpoint twice on development data.
    expected = [row["id"] for row in development[QUALIFICATION_TASK][1]]
    repeats = []
    for repeat in (1, 2):
        destination = EVALUATION_ROOT / "qualification" / label / f"repeat_{repeat}"
        collect(checkpoint, QUALIFICATION_TASK, QUALIFICATION_BUNDLE, destination)
        require_native(destination, QUALIFICATION_TASK, checkpoint, QUALIFICATION_BUNDLE)
        rows = [
            json.loads(line)
            for line in (destination / "predictions.jsonl").read_text().splitlines()
            if line.strip()
        ]
        if [row["id"] for row in rows] != expected:
            raise ValueError("Development prediction order differs from the source data.")
        if any(
            not isinstance(row.get("scores"), list)
            or len(row["scores"]) != 3
            or any(
                type(score) not in (int, float) or not math.isfinite(score)
                for score in row["scores"]
            )
            for row in rows
        ):
            raise ValueError("Expected three finite candidate scores per development item.")
        repeats.append(rows)
    difference = max(
        abs(a - b)
        for first, second in zip(*repeats, strict=True)
        for a, b in zip(first["scores"], second["scores"], strict=True)
    )
    proof = {
        "checkpoint": str(checkpoint),
        "development_manifest_sha256": sha256(QUALIFICATION_BUNDLE),
        "absolute_tolerance": ABSOLUTE_TOLERANCE,
        "max_absolute_score_difference": difference,
        "passed": difference <= ABSOLUTE_TOLERANCE,
    }
    save_json(EVALUATION_ROOT / "qualification" / label / "proof.json", proof)
    if not proof["passed"]:
        raise ValueError("Development scores differ by more than ABSOLUTE_TOLERANCE.")

## 1. Evaluate before unlearning

Enable `RUN_BEFORE` to evaluate the six base models. A repeatability check runs each unchanged checkpoint twice on development data before evaluation. Each task is then collected once per model.

In [ ]:
if RUN_BEFORE:
    full, development = preflight()
    for model in MODELS:
        checkpoint = BASE_CHECKPOINTS[model["id"]]
        require_checkpoint(checkpoint)
        if sha256(checkpoint / "config.json") != model["config_sha256"]:
            raise ValueError(f"Pinned base config differs: {model['id']}")
    for model in MODELS:
        qualify(BASE_CHECKPOINTS[model["id"]], "before/" + model["id"], development)
    for model in MODELS:
        for task in TASKS:
            destination = EVALUATION_ROOT / "before" / model["id"] / task
            collect(BASE_CHECKPOINTS[model["id"]], task, EVALUATION_BUNDLE, destination)
            require_native(destination, task, BASE_CHECKPOINTS[model["id"]], EVALUATION_BUNDLE)
else:
    print(
        "Before collection is disabled. Enable RUN_BEFORE when evaluation inputs and GPUs are ready."
    )

## 2. Evaluate after unlearning

Train with the [method notebooks](README.md), then enable `RUN_AFTER`. The notebook checks all 30 checkpoint exports and training receipts, verifies the baseline runs and repeats the development check before collecting post-training measurements. Task vectors are evaluated at their exported checkpoint paths, including when the selected scale is zero.

In [ ]:
if RUN_AFTER:
    full, development = preflight()
    training_evidence = {}
    for model in MODELS:
        for task in TASKS:
            require_native(
                EVALUATION_ROOT / "before" / model["id"] / task,
                task,
                BASE_CHECKPOINTS[model["id"]],
                EVALUATION_BUNDLE,
            )
        for method in METHODS:
            training_evidence[method + "/" + model["id"]] = require_training(method, model)
    EVALUATION_ROOT.mkdir(parents=True, exist_ok=True)
    evidence_path = EVALUATION_ROOT / "training_evidence.json"
    if evidence_path.exists():
        raise FileExistsError("Training evidence already exists. Choose a new evaluation root.")
    save_json(evidence_path, training_evidence)
    for method in METHODS:
        for model in MODELS:
            qualify(
                AFTER_CHECKPOINTS[method, model["id"]],
                "after/" + method + "/" + model["id"],
                development,
            )
    for method in METHODS:
        for model in MODELS:
            for task in TASKS:
                checkpoint = AFTER_CHECKPOINTS[method, model["id"]]
                destination = EVALUATION_ROOT / "after" / method / model["id"] / task
                collect(checkpoint, task, EVALUATION_BUNDLE, destination)
                require_native(destination, task, checkpoint, EVALUATION_BUNDLE)
else:
    print("After collection is disabled. Enable RUN_AFTER once the method checkpoints are saved.")

## 3. Compare before and after

Enable `RUN_COMPARISON` to compare saved predictions on the CPU. Fair-UK returns native metric values, paired changes, confidence intervals and availability reasons.

In [ ]:
if RUN_COMPARISON:
    preflight()
    for model in MODELS:
        for task in TASKS:
            require_native(
                EVALUATION_ROOT / "before" / model["id"] / task,
                task,
                BASE_CHECKPOINTS[model["id"]],
                EVALUATION_BUNDLE,
            )
        for method in METHODS:
            require_training(method, model)
            for task in TASKS:
                require_native(
                    EVALUATION_ROOT / "after" / method / model["id"] / task,
                    task,
                    AFTER_CHECKPOINTS[method, model["id"]],
                    EVALUATION_BUNDLE,
                )
    for method in METHODS:
        for model in MODELS:
            for task in TASKS:
                destination = EVALUATION_ROOT / "paired" / method / model["id"] / task
                invoke(
                    [
                        "paper",
                        "compare-native",
                        "--before-run",
                        str(EVALUATION_ROOT / "before" / model["id"] / task),
                        "--after-run",
                        str(EVALUATION_ROOT / "after" / method / model["id"] / task),
                        "--dataset-bundle",
                        str(EVALUATION_BUNDLE),
                        "--bootstrap",
                        str(BOOTSTRAP),
                        "--seed",
                        str(SEED),
                        "--output",
                        str(destination),
                    ],
                    destination,
                    gpu=False,
                )
else:
    print(
        "Saved-run comparison is disabled. Enable RUN_COMPARISON after before and after collection."
    )

## Results

Read the Fair-UK `paired.json` reports and export their values to `native_before_after.csv`. The CSV includes all rows; the display shows the first 100. Predictions, native reports and paired reports remain in the output directories.

In [ ]:
summary = []
for method in METHODS:
    for model in MODELS:
        for task in TASKS:
            report_path = EVALUATION_ROOT / "paired" / method / model["id"] / task / "paired.json"
            if not report_path.is_file():
                continue
            report = read_json(report_path)
            if (
                report.get("kind") != "paired_native_checkpoint_evaluation"
                or report.get("task") != task
            ):
                raise ValueError(f"Unexpected paired report: {report_path}")
            for effect in report["effects"]:
                summary.append({"model": model["id"], "method": method, "task": task, **effect})
if summary:
    columns = [
        "model",
        "method",
        "task",
        "scope",
        "metric",
        "unit",
        "before",
        "after",
        "after_minus_before",
        "ci95",
        "status",
        "uncertainty_status",
        "reason",
    ]
    summary_path = EVALUATION_ROOT / "native_before_after.csv"
    with summary_path.open("w", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=columns, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(summary)
    header = "".join("<th>" + html.escape(column) + "</th>" for column in columns)
    body = "".join(
        "<tr>"
        + "".join("<td>" + html.escape(str(row.get(column, ""))) + "</td>" for column in columns)
        + "</tr>"
        for row in summary[:100]
    )
    display(
        HTML("<table><thead><tr>" + header + "</tr></thead><tbody>" + body + "</tbody></table>")
    )
    print("All native result rows:", len(summary), "| CSV:", summary_path)
else:
    print("No paired reports have been collected in this evaluation root yet.")